# Llama-3.2-3B-Instruct Fine-tuning — Clasificador Inverso (Anti-Humor)
## Clasificacion de **NO humor** en Espanol — Clasificador Complementario para el Ensamble

### Motivacion

Este notebook entrena un clasificador **inverso**: en lugar de detectar humor, detecta **ausencia de humor (texto serio/neutro)**.
Esto aporta una perspectiva ortogonal al ensamble porque:
- Los errores del clasificador directo (FP: texto serio clasificado como humor) son los aciertos de este clasificador
- El score de 'no humor' captura senales linguisticas distintas (literalidad, ausencia de script-switch OSTH)
- Llama 3.2-3B tiene un preentrenamiento diferente a Qwen/Gemma, por lo que sus errores son independientes

### Diferencia clave: Objetivo Invertido

| Aspecto | Pipeline Original (Qwen OSTH) | **Este notebook (Llama Anti-Humor)** |
|---|---|---|
| Modelo | Qwen2.5-3B-Instruct | **Llama-3.2-3B-Instruct** |
| Objetivo | Predecir **humor** (Si=humor) | Predecir **NO humor** (Si=texto serio) |
| Label map | 1->'Si', 0->'No' | **0->'Si' (serio), 1->'No' (humoristico)** |
| Chat template | `<|im_start|>` | **`<|begin_of_text|>` / `<|eot_id|>`** |
| Response template | `<|im_start|>assistant\n` | **`<|start_header_id|>assistant<|end_header_id|>\n\n`** |
| Score para ensamble | `p_humor` | **`1 - p_serio` (invertido)** |
| Objetivo en ensamble | Clasificador con sesgo OSTH | **Clasificador inverso - diversidad de error** |

### Por que funciona en el ensamble

El ensamble promedia scores de `p_humor`. Este modelo produce `score_antih = 1 - p_serio`,
que es numericamente equivalente a un score de humor pero calculado desde la perspectiva opuesta.
Cuando el modelo dice 'esto NO es serio' con alta confianza, aporta evidencia independiente de humor.


## ── Sección 0: Instalación ───────────────────────────────────────────────────

In [ ]:
!pip install -q \
    transformers>=4.47.0 \
    datasets==3.2.0 \
    peft==0.14.0 \
    trl==0.13.0 \
    bitsandbytes>=0.46.1 \
    accelerate==1.2.1 \
    scikit-learn \
    evaluate

!pip install -q --upgrade transformers accelerate safetensors

print('✅ Dependencias instaladas')

## ── Sección 1: Autenticación y verificación de GPU ──────────────────────────

In [ ]:
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print('✅ Autenticado con Colab Secrets')
except Exception:
    login()

In [ ]:
import torch

print(f'GPU disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('❌ Se requiere GPU. Cambia el runtime a GPU en Colab.')

## ── Sección 2: Configuración central ───────────────────────────────────────

In [ ]:
import os

CFG = dict(
    # Modelo
    model_id        = 'meta-llama/Llama-3.2-3B-Instruct',

    # Datos (mismos archivos que el pipeline original)
    train_file      = 'dataset_humor_train.json',
    val_file        = None,
    test_file       = 'dataset_humor_test.json',
    text_col        = 'text',
    label_col       = 'klass',

    # OBJETIVO INVERTIDO
    # label_map INVERTIDO: 0 (no humor) -> 'Si' (es serio), 1 (humor) -> 'No' (no es serio)
    # El modelo aprende a detectar AUSENCIA de humor.
    label_map       = {0: 'Si', 1: 'No'},   # <-- clave: invertido respecto al original
    positive_label  = 0,                      # clase "positiva" para este modelo = no humor

    # QLoRA
    # Llama-3.2-3B comparte arquitectura GQA con mismos target_modules que Llama 3
    lora_r          = 16,
    lora_alpha      = 32,
    lora_dropout    = 0.05,
    target_modules  = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                       'gate_proj', 'up_proj', 'down_proj'],

    # Entrenamiento
    max_seq_len     = 256,
    batch_size      = 8,
    grad_accum      = 4,    # effective batch = 32
    epochs          = 5,
    lr              = 2e-4,
    warmup_ratio    = 0.08,
    weight_decay    = 0.01,
    seed            = 42,

    # Salida
    output_dir      = './llama3b-antihumor-qlora',
)

os.makedirs(CFG['output_dir'], exist_ok=True)
print('Configuracion cargada (Objetivo: Anti-Humor / Clasificador Inverso):')
for k, v in CFG.items():
    print(f'  {k:20s} = {v}')
print()
print('RECORDATORIO: label_map INVERTIDO => 0 (no humor)=Si | 1 (humor)=No')


## ── Sección 3: Carga y split del dataset ───────────────────────────────────

In [ ]:
import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

df = pd.read_json(CFG['train_file'], lines=True)
X  = df[CFG['text_col']].values
y  = df[CFG['label_col']].values

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size    = 0.10,
    random_state = CFG['seed'],
    stratify     = y,
)

train_df = pd.DataFrame({CFG['text_col']: X_train, CFG['label_col']: y_train})
val_df   = pd.DataFrame({CFG['text_col']: X_val,   CFG['label_col']: y_val})

raw = DatasetDict({
    'train':      Dataset.from_pandas(train_df),
    'validation': Dataset.from_pandas(val_df),
})

if CFG['test_file']:
    test_df   = pd.read_json(CFG['test_file'], lines=True)
    raw['test'] = Dataset.from_pandas(test_df)

print(f'Train:      {len(raw["train"]):,} ejemplos')
print(f'Validation: {len(raw["validation"]):,} ejemplos')
if 'test' in raw:
    print(f'Test:       {len(raw["test"]):,} ejemplos')

In [ ]:
# ── Distribución de clases y class weights ────────────────────────────────────
labels_train = raw['train'][CFG['label_col']]
unique, counts = np.unique(labels_train, return_counts=True)

print(f'Train: {len(raw["train"]):,} | Val: {len(raw["validation"]):,}')
print('Distribución train:')
for u, c in zip(unique, counts):
    print(f'   Clase {u} ({CFG["label_map"].get(u, u)}): {c:,} ({100*c/len(labels_train):.1f}%)')

class_counts  = dict(zip(unique, counts))
total         = sum(class_counts.values())
CLASS_WEIGHTS = {k: total / (len(class_counts) * v) for k, v in class_counts.items()}
print(f'\nClass weights: {CLASS_WEIGHTS}')

## Seccion 4: Tokenizador y template de prompt Anti-Humor

### Diferencia critica: Llama 3.2 usa `<|begin_of_text|>` y `<|eot_id|>`

El `RESPONSE_TEMPLATE` debe coincidir **exactamente** con el header del turno
assistant en Llama 3.2. Si no coincide, `DataCollatorForCompletionOnlyLM` entrena
sobre todo el prompt en lugar de solo la respuesta (Si/No).

Llama 3.2 usa el mismo chat template que Llama 3.1:
```
<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{system}<|eot_id|>
<|start_header_id|>user<|end_header_id|>\n\n{user}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>\n\n{respuesta}<|eot_id|>
```
Por lo tanto: `RESPONSE_TEMPLATE = '<|start_header_id|>assistant<|end_header_id|>\n\n'`


In [ ]:
from transformers import AutoTokenizer

print('Cargando tokenizador Llama-3.2-3B-Instruct...')
print('(Llama 3.2 requiere acceso aprobado en HuggingFace -- verifica tu token)')
tokenizer = AutoTokenizer.from_pretrained(CFG['model_id'])
tokenizer.padding_side = 'left'   # critico para generacion correcta en Llama

# Llama 3.2 no tiene pad_token por defecto -> usar eos_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    print('AVISO: pad_token no encontrado -> usando eos_token como pad_token')

# Detectar automaticamente el response template de Llama 3.2
dummy_messages = [
    {'role': 'system',    'content': 'eres un asistente'},
    {'role': 'user',      'content': 'hola'},
    {'role': 'assistant', 'content': 'hola'},
]
dummy_prompt = tokenizer.apply_chat_template(
    dummy_messages, tokenize=False, add_generation_prompt=False
)
print('\nFormato del chat template de Llama 3.2:')
print(repr(dummy_prompt))
print()

# Response template para Llama 3.x
RESPONSE_TEMPLATE = '<|start_header_id|>assistant<|end_header_id|>\n\n'
assert RESPONSE_TEMPLATE in dummy_prompt, (
    f'ERROR: RESPONSE_TEMPLATE no encontrado en el chat template.\n'
    f'Prompt dummy: {repr(dummy_prompt)}\n'
    f'Ajusta RESPONSE_TEMPLATE manualmente.'
)
print(f'OK RESPONSE_TEMPLATE verificado: {repr(RESPONSE_TEMPLATE)}')


In [ ]:
# ── System prompt Anti-Humor (perspectiva inversa) ───────────────────────────
# La instruccion esta INVERTIDA: el modelo busca caracteristicas de texto SERIO,
# es decir, la AUSENCIA de los mecanismos OSTH de humor.
# Esto hace que el modelo aprenda representaciones desde el angulo opuesto,
# capturando senales de literalidad, coherencia semantica y ausencia de punch line.

ANTI_HUMOR_SYSTEM_PROMPT = (
    "Eres un experto en linguistica computacional del humor "
    "especializado en la Teoria Semantica Ontologica del Humor (OSTH).\n"
    "Tu tarea es identificar tweets que NO son humoristicos, es decir, texto serio, literal o neutro.\n"
    "Un tweet es SERIO (no humoristico) cuando:\n"
    "- No presenta cambio de guion semantico (script switch): el contexto es coherente de principio a fin.\n"
    "- No hay punto de disparo (punch line): no existe palabra o frase que active un giro inesperado.\n"
    "- No hay oposicion del tipo normal/anormal, real/irreal o posible/imposible.\n"
    "- El mensaje es directo, literal y su interpretacion principal no cambia al releerlo.\n"
    "Responde UNICAMENTE con la palabra 'Si' (el tweet es serio, no contiene humor) "
    "o 'No' (el tweet contiene humor)."
)


def make_prompt(tweet: str, include_answer: bool = False, label: int = None) -> str:
    """
    Genera el prompt usando el chat template nativo de Llama 3.2.
    IMPORTANTE: label es el label ORIGINAL del dataset (1=humor, 0=no humor).
    La inversion se aplica internamente via CFG['label_map']:
      label 0 (no humor) -> respuesta 'Si' (es serio)
      label 1 (humor)    -> respuesta 'No' (no es serio)
    - include_answer=True  -> para entrenamiento (incluye la respuesta invertida)
    - include_answer=False -> para inferencia
    """
    messages = [
        {'role': 'system', 'content': ANTI_HUMOR_SYSTEM_PROMPT},
        {'role': 'user',   'content': f'Tweet a analizar: "{tweet}"'},
    ]
    if include_answer and label is not None:
        # CFG['label_map'] ya esta invertido: {0: 'Si', 1: 'No'}
        answer = CFG['label_map'].get(int(label), 'No')
        messages.append({'role': 'assistant', 'content': answer})
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
    else:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )


def make_training_example(sample: dict) -> dict:
    label_value = sample.get(CFG['label_col'], None)
    return {'text': make_prompt(
        sample[CFG['text_col']],
        include_answer = True,
        label          = label_value,
    )}


# Aplicar template a todos los splits
print('Mapeando splits con objetivo invertido (Anti-Humor)...')
dataset = raw.map(make_training_example, remove_columns=raw['train'].column_names)

# Verificacion del prompt
print('\nEjemplo de prompt de entrenamiento (Anti-Humor):')
print('-' * 60)
print(dataset['train'][0]['text'])
print('-' * 60)

# Verificar inversion: un tweet con label=1 (humor) debe terminar en 'No'
sample_idx = next(i for i, x in enumerate(raw['train'][CFG['label_col']]) if x == 1)
example_humor = dataset['train'][sample_idx]['text']
assert example_humor.strip().endswith('No'), (
    'ERROR: Un tweet de humor (label=1) deberia terminar en "No" (no es serio).\n'
    f'Texto final: {example_humor[-50:]}'
)
sample_idx0 = next(i for i, x in enumerate(raw['train'][CFG['label_col']]) if x == 0)
example_serio = dataset['train'][sample_idx0]['text']
assert example_serio.strip().endswith('Si'), (
    'ERROR: Un tweet serio (label=0) deberia terminar en "Si" (es serio).\n'
    f'Texto final: {example_serio[-50:]}'
)
print('OK Inversion de etiquetas verificada:')
print('   label=1 (humor)    -> respuesta: "No"  (no es serio)')
print('   label=0 (no humor) -> respuesta: "Si"  (es serio)')

# Verificar que el response template esta presente
example = dataset['train'][0]['text']
assert RESPONSE_TEMPLATE in example, 'ERROR: RESPONSE_TEMPLATE no aparece en el ejemplo.'
print(f'OK RESPONSE_TEMPLATE presente: {repr(RESPONSE_TEMPLATE)}')

tokens = tokenizer(example, return_tensors='pt')
print(f'Tokens en ejemplo: {tokens["input_ids"].shape[1]} (max_seq_len={CFG["max_seq_len"]})')
if tokens['input_ids'].shape[1] > CFG['max_seq_len']:
    print('AVISO: El ejemplo supera max_seq_len -- considera aumentar a 320')


In [ ]:
# ── Análisis de longitud de tokens en el dataset completo ────────────────────
print('Analizando longitud de tokens en el dataset de entrenamiento...')

lengths = [
    tokenizer(ex, return_tensors='pt')['input_ids'].shape[1]
    for ex in dataset['train']['text'][:200]   # muestra de 200
]
lengths = np.array(lengths)

print(f'  Media:    {lengths.mean():.0f} tokens')
print(f'  Mediana:  {np.median(lengths):.0f} tokens')
print(f'  P95:      {np.percentile(lengths, 95):.0f} tokens')
print(f'  Máximo:   {lengths.max()} tokens')
print(f'  > max_seq_len ({CFG["max_seq_len"]}): {(lengths > CFG["max_seq_len"]).mean()*100:.1f}%')

# Si más del 5% de ejemplos supera max_seq_len, ajustamos automáticamente
p99 = int(np.percentile(lengths, 99))
if p99 > CFG['max_seq_len']:
    print(f'\n⚠️  Ajustando max_seq_len de {CFG["max_seq_len"]} → {p99} (P99)')
    CFG['max_seq_len'] = p99
else:
    print(f'\n✅ max_seq_len={CFG["max_seq_len"]} cubre el P99 ({p99} tokens)')

## ── Sección 5: Carga del modelo base con QLoRA ──────────────────────────────

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

print('Cargando Llama-3.2-3B-Instruct con cuantizacion NF4...')
print('(Llama 3.2 requiere acceso aprobado en HuggingFace -- verifica tu token HF_TOKEN)')

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = 'nf4',
    bnb_4bit_compute_dtype    = torch.bfloat16,
    bnb_4bit_use_double_quant = True,
)

model = AutoModelForCausalLM.from_pretrained(
    CFG['model_id'],
    quantization_config  = bnb_config,
    device_map           = 'auto',
    torch_dtype          = torch.bfloat16,
    attn_implementation  = 'eager',   # usar 'flash_attention_2' si la GPU lo soporta (A100/H100)
)
model.config.use_cache = False   # necesario para gradient checkpointing

mem = torch.cuda.memory_allocated() / 1e9
print(f'\nOK Modelo cargado -- VRAM usada: {mem:.2f} GB')
print(f'   Parametros totales: {sum(p.numel() for p in model.parameters()):,}')
print(f'   Arquitectura: Llama-3.2 (GQA, RoPE, SwiGLU)')


## ── Sección 6: Configuración de LoRA ───────────────────────────────────────

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r              = CFG['lora_r'],
    lora_alpha     = CFG['lora_alpha'],
    target_modules = CFG['target_modules'],
    lora_dropout   = CFG['lora_dropout'],
    bias           = 'none',
    task_type      = 'CAUSAL_LM',
)

model = get_peft_model(model, lora_config)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parámetros totales:      {total_params:,}')
print(f'Parámetros entrenables:  {trainable_params:,} ({100*trainable_params/total_params:.2f}%)')
print(f'lora_r={CFG["lora_r"]} | lora_alpha={CFG["lora_alpha"]} | dropout={CFG["lora_dropout"]}')

## ── Sección 7: Entrenamiento con SFTTrainer ─────────────────────────────────

In [ ]:
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM
from torch.utils.data import WeightedRandomSampler

# ── Collator: solo penaliza los tokens de respuesta (Sí / No) ────────────────
collator = DataCollatorForCompletionOnlyLM(
    response_template = RESPONSE_TEMPLATE,
    tokenizer         = tokenizer,
    mlm               = False,
)

# ── WeightedRandomSampler para balancear clases ───────────────────────────────
sample_weights = [
    CLASS_WEIGHTS[int(lbl)]
    for lbl in raw['train'][CFG['label_col']]
]
sampler = WeightedRandomSampler(
    weights     = sample_weights,
    num_samples = len(sample_weights),
    replacement = True,
)

# ── Argumentos de entrenamiento ───────────────────────────────────────────────
training_args = SFTConfig(
    output_dir                    = CFG['output_dir'],
    num_train_epochs              = CFG['epochs'],
    per_device_train_batch_size   = CFG['batch_size'],
    per_device_eval_batch_size    = CFG['batch_size'],
    gradient_accumulation_steps   = CFG['grad_accum'],
    optim                         = 'paged_adamw_8bit',
    learning_rate                 = CFG['lr'],
    lr_scheduler_type             = 'cosine',
    warmup_ratio                  = CFG['warmup_ratio'],
    weight_decay                  = CFG['weight_decay'],
    bf16                          = True,
    gradient_checkpointing        = True,
    gradient_checkpointing_kwargs = {'use_reentrant': False},
    max_seq_length                = CFG['max_seq_len'],
    packing                       = False,
    dataset_text_field            = 'text',
    eval_strategy                 = 'epoch',
    save_strategy                 = 'epoch',
    load_best_model_at_end        = True,
    metric_for_best_model         = 'eval_loss',
    greater_is_better             = False,
    save_total_limit              = 2,
    logging_steps                 = 50,
    report_to                     = 'none',
    seed                          = CFG['seed'],
)

trainer = SFTTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = dataset['train'],
    eval_dataset     = dataset['validation'],
    processing_class = tokenizer,
    data_collator    = collator,
)

print('✅ Trainer configurado')
print(f'   RESPONSE_TEMPLATE   = {repr(RESPONSE_TEMPLATE)}')
print(f'   padding_side        = {tokenizer.padding_side}')
print(f'   packing             = False')
print(f'   lora_r              = {CFG["lora_r"]}')
print(f'   max_seq_len         = {CFG["max_seq_len"]}')
print(f'   class_weights       = {CLASS_WEIGHTS}')

In [ ]:
# ── Entrenamiento ─────────────────────────────────────────────────────────────
print('Iniciando entrenamiento...')
train_result = trainer.train()

print('\n── Métricas de entrenamiento ──────────────────────────────')
for k, v in train_result.metrics.items():
    print(f'  {k}: {v:.4f}')

## ── Sección 8: Inferencia y evaluación ─────────────────────────────────────

In [ ]:
from sklearn.metrics import classification_report, f1_score, confusion_matrix

# Encontrar IDs de token para 'Si' y 'No' en Llama 3.2
# Llama 3.2 tokeniza de forma diferente a Qwen/Gemma -> verificar siempre
SI_IDS = tokenizer.encode('Si', add_special_tokens=False)
NO_IDS = tokenizer.encode('No', add_special_tokens=False)

print(f'Tokens para "Si": {SI_IDS} -> {[tokenizer.decode([t]) for t in SI_IDS]}')
print(f'Tokens para "No": {NO_IDS} -> {[tokenizer.decode([t]) for t in NO_IDS]}')

# Si alguna palabra se tokeniza en multiples tokens, usamos el primero
SI_TOKEN_ID = SI_IDS[0]
NO_TOKEN_ID = NO_IDS[0]

assert SI_TOKEN_ID != NO_TOKEN_ID, 'ERROR: Si y No comparten el mismo token ID en Llama 3.2'
print(f'\nOK SI_TOKEN_ID={SI_TOKEN_ID} (= serio/no_humor) | NO_TOKEN_ID={NO_TOKEN_ID} (= humor)')
print()
print('Interpretacion de los scores de este modelo:')
print('   p_serio     = P(token="Si") = probabilidad de que el tweet sea SERIO')
print('   score_antih = 1 - p_serio   = probabilidad de humor (para el ensamble)')


In [ ]:
def predict_batch(tweets: list, batch_size: int = 16) -> tuple:
    """
    Inferencia Anti-Humor por comparacion de logits (Si=serio vs No=humor).

    IMPORTANTE -- Inversion de scores para el ensamble:
    - Este modelo predice P(tweet es SERIO) = p_serio
    - Para el ensamble necesitamos P(tweet es HUMOR) = 1 - p_serio
    - Retorna:
        predictions_humor : prediccion binaria en escala original (1=humor, 0=no humor)
        scores_antih      : 1 - p_serio, interpretable como score de humor
    """
    model.eval()
    predictions_humor, scores_antih = [], []

    for i in range(0, len(tweets), batch_size):
        batch   = tweets[i: i + batch_size]
        prompts = [make_prompt(t, include_answer=False) for t in batch]

        inputs = tokenizer(
            prompts,
            return_tensors = 'pt',
            padding        = True,
            truncation     = True,
            max_length     = CFG['max_seq_len'],
        ).to(model.device)

        with torch.no_grad():
            outputs     = model(**inputs)
            last_logits = outputs.logits[:, -1, :]        # (B, vocab)
            si_logit    = last_logits[:, SI_TOKEN_ID]      # (B,) -- logit de "serio"
            no_logit    = last_logits[:, NO_TOKEN_ID]      # (B,) -- logit de "humor"

            # p_serio: probabilidad de que el tweet sea serio
            p_serio = torch.softmax(
                torch.stack([no_logit, si_logit], dim=1), dim=1
            )[:, 1]

            # Invertir para el ensamble: score_antih = 1 - p_serio ~ p_humor
            score_antih = 1.0 - p_serio

            # Prediccion en escala original (1=humor, 0=no humor)
            # El modelo dice "No" (no es serio) -> prediccion humor = 1
            pred_humor = (no_logit > si_logit).int()

            predictions_humor.extend(pred_humor.cpu().tolist())
            scores_antih.extend(score_antih.cpu().tolist())

    return predictions_humor, scores_antih


print('OK Funcion predict_batch (Anti-Humor) definida')
print('   Retorna: (pred_humor [0/1], score_antih [0..1 ~ p_humor])')


In [ ]:
# Evaluacion en validacion
print('Evaluando en validacion (Anti-Humor -- metricas en escala original humor/no-humor)...')
val_tweets = list(raw['validation'][CFG['text_col']])
val_labels = np.array(raw['validation'][CFG['label_col']])  # 0=no humor, 1=humor (original)

val_preds, val_scores = predict_batch(val_tweets)
val_scores_np = np.array(val_scores)   # score_antih = 1 - p_serio

# Metricas en escala original (humor=1, no_humor=0)
print('\n-- Resultados con umbral 0.5 (escala original humor/no-humor) ------')
print(classification_report(val_labels, val_preds,
                             target_names=['No humor', 'Humor']))
print('Matriz de Confusion:')
print(confusion_matrix(val_labels, val_preds))


In [ ]:
# Busqueda del umbral optimo en validacion
# Nota: score_antih ~ p_humor, umbral > 0.5 -> predice humor
best_thresh, best_f1 = 0.5, 0.0

for thresh in np.linspace(0.1, 0.9, 81):
    preds_t = (val_scores_np >= thresh).astype(int)
    f1_t    = f1_score(val_labels, preds_t, average='macro', zero_division=0)
    if f1_t > best_f1:
        best_f1, best_thresh = f1_t, thresh

val_preds_calibrated = (val_scores_np >= best_thresh).astype(int)

print(f'Umbral default  (0.50): F1 Macro = {f1_score(val_labels, val_preds, average="macro"):.4f}')
print(f'Umbral calibrado({best_thresh:.2f}): F1 Macro = {best_f1:.4f}  <- usar este en test')
print('\n-- Con umbral calibrado -----------------------------------------')
print(classification_report(val_labels, val_preds_calibrated,
                             target_names=['No humor', 'Humor']))


In [ ]:
# Distribucion de scores_antih -- diagnostico de calibracion
# score_antih = 1 - p_serio -> si std > 0.20, el modelo discrimina bien
print('-- Distribucion de score_antih (= 1 - p_serio ~ p_humor) ---------------')
print(f'  Media:    {val_scores_np.mean():.3f}')
print(f'  Std:      {val_scores_np.std():.3f}   <- objetivo > 0.20')
print(f'  P10:      {np.percentile(val_scores_np, 10):.3f}')
print(f'  P50:      {np.percentile(val_scores_np, 50):.3f}')
print(f'  P90:      {np.percentile(val_scores_np, 90):.3f}')

humor_scores    = val_scores_np[val_labels == 1]
no_humor_scores = val_scores_np[val_labels == 0]
print(f'\n  Clase Humor    -> media score_antih: {humor_scores.mean():.3f}  (esperado: alto)')
print(f'  Clase No humor -> media score_antih: {no_humor_scores.mean():.3f}  (esperado: bajo)')
print(f'  Separacion:                          {humor_scores.mean() - no_humor_scores.mean():.3f}')
print()
print('Si score_antih de "No humor" es bajo y de "Humor" es alto -> inversion correcta')


## ── Sección 9: Análisis de errores ─────────────────────────────────────────

In [ ]:
val_df_err = pd.DataFrame({
    'tweet'    : val_tweets,
    'true'     : val_labels,
    'pred'     : val_preds_calibrated,
    'score_si' : val_scores,
})
val_df_err['error']      = val_df_err['true'] != val_df_err['pred']
val_df_err['error_type'] = ''
val_df_err.loc[(val_df_err['true']==1) & (val_df_err['pred']==0), 'error_type'] = 'FN'
val_df_err.loc[(val_df_err['true']==0) & (val_df_err['pred']==1), 'error_type'] = 'FP'

print('── Errores por tipo ──────────────────────────────────────────')
print(val_df_err[val_df_err['error']]['error_type'].value_counts().to_string())

print('\n── Falsos Negativos (humor no detectado) ─────────────────────')
fn = val_df_err[val_df_err['error_type'] == 'FN'].sort_values('score_si')
for _, row in fn.head(10).iterrows():
    print(f'  score={row["score_si"]:.3f} | {row["tweet"][:80]}')

print('\n── Falsos Positivos (no humor clasificado como humor) ────────')
fp = val_df_err[val_df_err['error_type'] == 'FP'].sort_values('score_si', ascending=False)
for _, row in fp.head(10).iterrows():
    print(f'  score={row["score_si"]:.3f} | {row["tweet"][:80]}')

## ── Sección 10: Evaluación en test ──────────────────────────────────────────

In [ ]:
if 'test' in raw:
    print(f'Evaluando en test ({len(raw["test"])} ejemplos)...')
    test_tweets = list(raw['test'][CFG['text_col']])

    test_preds_raw, test_scores = predict_batch(test_tweets)
    test_scores_np = np.array(test_scores)   # score_antih

    # Aplicar umbral calibrado en validacion
    test_preds_final = (test_scores_np >= best_thresh).astype(int)

    print(f'Umbral aplicado: {best_thresh:.2f}')
    unique, counts = np.unique(test_preds_final, return_counts=True)
    print(f'Distribucion predicciones (escala original): {dict(zip(unique.tolist(), counts.tolist()))}')
else:
    print('No hay split de test')


## ── Sección 11: Guardado del adaptador y resultados ─────────────────────────

In [ ]:
import json

# Guardar adaptador LoRA
adapter_path = os.path.join(CFG['output_dir'], 'best_adapter')
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f'OK Adaptador guardado: {adapter_path}')

# Metadata del experimento
meta = {
    'modelo'           : CFG['model_id'],
    'experimento'      : 'Llama-3.2-3B QLoRA -- Clasificador Inverso Anti-Humor',
    'objetivo'         : 'Predice NO humor (Si=serio, No=humor) -- score_antih = 1 - p_serio',
    'label_map'        : {str(k): v for k, v in CFG['label_map'].items()},
    'lora_r'           : CFG['lora_r'],
    'lora_alpha'       : CFG['lora_alpha'],
    'epochs'           : CFG['epochs'],
    'lr'               : CFG['lr'],
    'max_seq_len'      : CFG['max_seq_len'],
    'threshold'        : float(best_thresh),
    'f1_macro_val'     : float(best_f1),
    'class_weights'    : {str(k): float(v) for k, v in CLASS_WEIGHTS.items()},
    'score_std_val'    : float(val_scores_np.std()),
    'separacion_clases': float(humor_scores.mean() - no_humor_scores.mean()),
    'response_template': RESPONSE_TEMPLATE,
    'nota_ensamble'    : 'Usar score_antih (= 1 - p_serio) directamente como score de humor',
}

meta_path = os.path.join(adapter_path, 'experiment_meta.json')
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print('\n-- Metadata del experimento ------------------------------------------')
print(json.dumps(meta, indent=2, ensure_ascii=False))


In [ ]:
# Guardar predicciones y scores
if 'test' in raw:
    # CSV de submission (formato concurso -- predicciones en escala original)
    submission_df = pd.DataFrame({
        'id'   : range(1, len(test_preds_final) + 1),
        'klass': test_preds_final,
    })
    submission_path = os.path.join(CFG['output_dir'], 'Llama_AntiHumor_submission.csv')
    submission_df.to_csv(submission_path, index=False)

    # CSV de scores para el ensamble
    # score_antih ya esta en escala p_humor -> puede usarse directamente
    scores_df = pd.DataFrame({
        'id'          : range(1, len(test_preds_final) + 1),
        'klass'       : test_preds_final,
        'score_antih' : test_scores_np,   # = 1 - p_serio ~ p_humor
    })
    scores_path = os.path.join(CFG['output_dir'], 'Llama_AntiHumor_scores_test.csv')
    scores_df.to_csv(scores_path, index=False)

# Scores de validacion (para re-optimizar pesos del ensamble)
val_scores_df = pd.DataFrame({
    'tweet'      : val_tweets,
    'true'       : val_labels,
    'pred'       : val_preds_calibrated,
    'score_antih': val_scores,   # = 1 - p_serio, en escala de humor
})
val_scores_path = os.path.join(CFG['output_dir'], 'Llama_AntiHumor_scores_val.csv')
val_scores_df.to_csv(val_scores_path, index=False)

print('OK Archivos guardados:')
if 'test' in raw:
    print(f'   {submission_path}')
    print(f'   {scores_path}')
print(f'   {val_scores_path}')


## ── Sección 12: Cómo integrar este modelo en el ensamble ───────────────────

Una vez entrenado, reemplaza el bloque de carga de Qwen en el notebook `Ensamble_Gemma_RoBERTuito_Qwen_OSTH.ipynb`:

In [ ]:
# CODIGO PARA COPIAR AL NOTEBOOK DEL ENSAMBLE (Seccion de carga de modelos)

ENSAMBLE_INTEGRATION_CODE = """
# ── Cargar Llama-3.2-3B Anti-Humor (clasificador inverso) ────────────────────
from peft import PeftModel

LLAMA_ADAPTER_PATH = "./llama3b-antihumor-qlora/best_adapter"

bnb_config_llama = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
llama_base = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-3B-Instruct",
    quantization_config=bnb_config_llama, device_map="auto",
)
llama_tokenizer = AutoTokenizer.from_pretrained(LLAMA_ADAPTER_PATH)
llama_tokenizer.padding_side = "left"
if llama_tokenizer.pad_token is None:
    llama_tokenizer.pad_token = llama_tokenizer.eos_token

llama_model = PeftModel.from_pretrained(llama_base, LLAMA_ADAPTER_PATH)
llama_model.eval()

LLAMA_SI_IDS = llama_tokenizer.encode("Si", add_special_tokens=False)
LLAMA_NO_IDS = llama_tokenizer.encode("No", add_special_tokens=False)
LLAMA_SI_TOKEN = LLAMA_SI_IDS[0]   # serio
LLAMA_NO_TOKEN = LLAMA_NO_IDS[0]   # humor

LLAMA_RESPONSE_TEMPLATE = "<|start_header_id|>assistant<|end_header_id|>\\n\\n"
ANTI_HUMOR_SYSTEM_PROMPT = (
    "Eres un experto en linguistica computacional del humor "
    "especializado en la Teoria Semantica Ontologica del Humor (OSTH).\\n"
    "Tu tarea es identificar tweets que NO son humoristicos, es decir, texto serio.\\n"
    "Responde UNICAMENTE con 'Si' (el tweet es serio) o 'No' (el tweet contiene humor)."
)


def make_llama_antih_prompt(tweet):
    messages = [
        {"role": "system", "content": ANTI_HUMOR_SYSTEM_PROMPT},
        {"role": "user",   "content": f'Tweet a analizar: "{tweet}"'},
    ]
    return llama_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def predict_llama_antih(tweets, batch_size=16):
    llama_model.eval()
    preds, scores = [], []
    for i in range(0, len(tweets), batch_size):
        batch = tweets[i: i + batch_size]
        prompts = [make_llama_antih_prompt(t) for t in batch]
        inputs = llama_tokenizer(
            prompts, return_tensors="pt", padding=True,
            truncation=True, max_length=256,
        ).to(llama_model.device)
        with torch.no_grad():
            logits = llama_model(**inputs).logits[:, -1, :]
            si_l = logits[:, LLAMA_SI_TOKEN]
            no_l = logits[:, LLAMA_NO_TOKEN]
            p_serio = torch.softmax(torch.stack([no_l, si_l], dim=1), dim=1)[:, 1]
            score_antih = 1.0 - p_serio          # en escala p_humor
            preds.extend((no_l > si_l).int().cpu().tolist())
            scores.extend(score_antih.cpu().tolist())
    return preds, scores


# En el bucle de ensamble:
# _, llama_scores_val  = predict_llama_antih(val_tweets)
# _, llama_scores_test = predict_llama_antih(test_tweets)
# score_ensamble = w1*score_gemma + w2*score_robertuito + w3*score_qwen + w4*score_antih
"""

print("Codigo de integracion Llama Anti-Humor para el ensamble:")
print("-" * 60)
print(ENSAMBLE_INTEGRATION_CODE)


---

## Checklist post-entrenamiento

Antes de integrar Llama Anti-Humor en el ensamble, verifica:

- [ ] `score_std_val > 0.20` -> el modelo discrimina bien en escala invertida
- [ ] `separacion_clases > 0.25` -> `score_antih` alto para humor, bajo para no-humor
- [ ] `F1 Macro val > 0.75` -> competitivo individualmente (criterio mas laxo: tarea inversa es mas dificil)
- [ ] La distribucion de `score_antih` es complementaria a la de `score_qwen` (correlacion < 0.85)
- [ ] Los FP del Qwen OSTH (textos serios clasificados como humor) tienen `score_antih` bajo

## Como usar los scores en el ensamble

```python
# Ensamble final con 4 modelos (optimizar w1..w4 en validacion con Optuna o GridSearch)
score_ensamble = (
    w1 * score_gemma       +   # Gemma fine-tuned (clasificador principal)
    w2 * score_robertuito  +   # RoBERTuito (encoder especializado espanol)
    w3 * score_qwen        +   # Qwen OSTH fine-tuned (sesgo OSTH directo)
    w4 * score_antih       +   # Llama Anti-Humor (perspectiva inversa)
)
pred_final = (score_ensamble >= umbral_optimo).astype(int)
```

**Por que `score_antih` aporta valor:**
- Entrenado desde el angulo opuesto -> sus errores son diferentes a los demas clasificadores
- Captura senales de literalidad que los clasificadores directos ignoran
- Llama 3.2 tiene un preentrenamiento distinto a Qwen/Gemma -> representaciones complementarias

## Proximo: Experimento 3 - Meta-clasificador con features OSTH

Con los cuatro clasificadores calibrados:

```python
X_meta = [
    score_gemma,       # score Gemma fine-tuned
    score_robertuito,  # score RoBERTuito
    score_qwen,        # score Qwen OSTH fine-tuned
    score_antih,       # score Llama Anti-Humor (invertido, ya en escala p_humor)
    tipo_oposicion_enc,
    oposicion_std_enc,
    tiene_punch_line,
    purview_complejidad,
]
```
